# =====================================================================
# GMFRMNet: Global-Local Multiscale Feature Refinement Mamba Network
# Remote Sensing Change Detection & Quantitative Area Calculation Pipeline
# =====================================================================

This notebook provides the complete end-to-end implementation for:
1. **Mounting Google Drive & Extracting Datasets** (HRSCD / Cropland datasets)
2. **Dataset Preprocessing & Tiling** ($10000\times10000$ or arbitrary sizes to $256\times256$ patches)
3. **GMFRMNet Deep Learning Model**: Weight-shared Siamese Visual Mamba (VMamba) Encoder + GLCSB (Global-Local Context State-Space Block) + MCSRB (Multiscale Channel-Spatial Refinement Block) Decoder
4. **Adaptive Weighted Loss** ($L_{Total} = \alpha L_{BCE} + \beta L_{Dice}$ / Multiclass)
5. **Full Training & Validation Loop** with Adam optimizer, Cosine Annealing, and standard metrics (OA, IoU, F1, Precision, Recall)
6. **INNOVATION: Area Quantification & Multiclass Transition Dynamics** (calculating exact change in $m^2$, hectares, and $km^2$ with land-cover transition matrices).

## Step 1: Environment Setup & Google Drive Mount

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install required geospatial & computer vision libraries
!pip install -q tifffile rasterio opencv-python albumentations einops

## Step 2: Extract & Verify Dataset from Google Drive
Configure your Google Drive paths below. If your dataset is stored as `.zip` files (such as `HRSCD.zip` or individual zips), this cell will automatically extract them.

In [ ]:
import os
import zipfile
import glob
from pathlib import Path

# Path to your HRSCD zip file in Google Drive
DRIVE_ZIP_PATH = '/content/drive/MyDrive/HRSCD.zip'
LOCAL_DATA_DIR = '/content/dataset_raw'
PATCHES_DIR = '/content/dataset_patches'

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(PATCHES_DIR, exist_ok=True)

# Auto-detect if file is named HRSCD.zip or HRSCD (without extension)
if not os.path.exists(DRIVE_ZIP_PATH) and os.path.exists('/content/drive/MyDrive/HRSCD'):
    DRIVE_ZIP_PATH = '/content/drive/MyDrive/HRSCD'

print(f"Target archive path: {DRIVE_ZIP_PATH}")

if os.path.exists(DRIVE_ZIP_PATH):
    size_gb = os.path.getsize(DRIVE_ZIP_PATH) / (1024 ** 3)
    print(f"Found archive: {os.path.basename(DRIVE_ZIP_PATH)} ({size_gb:.2f} GB)")
    print(f"Extracting to {LOCAL_DATA_DIR}... Please wait.")
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
        zf.extractall(LOCAL_DATA_DIR)
    print(f"Extraction complete! Files extracted to: {LOCAL_DATA_DIR}")
else:
    # Fallback search if located in a subfolder or slightly different name
    print(f"Not found directly at {DRIVE_ZIP_PATH}. Searching MyDrive for 'HRSCD' archives...")
    matches = glob.glob('/content/drive/MyDrive/**/HRSCD*.zip', recursive=True) + [
        p for p in glob.glob('/content/drive/MyDrive/**/HRSCD*', recursive=True) if not os.path.isdir(p)
    ]
    if matches:
        chosen = matches[0]
        print(f"Found: {chosen}. Extracting to {LOCAL_DATA_DIR}...")
        with zipfile.ZipFile(chosen, 'r') as zf:
            zf.extractall(LOCAL_DATA_DIR)
        print("Extraction complete!")
    else:
        raise FileNotFoundError("HRSCD zip archive not found in Google Drive! Please check the file name and path in MyDrive.")

## Step 3: Patch Extraction & Preprocessing Pipeline ($256 \times 256$)
Large remote sensing tiles ($10,000 \times 10,000$) are sliced into $256 \times 256$ non-overlapping or overlapping patches, filtering out empty/all-black background regions.

In [ ]:
import re
import cv2
import numpy as np
import tifffile
from tqdm import tqdm

def extract_patch_dataset(data_dir, output_dir, patch_size=256, max_tiles=20, min_change_ratio=0.01):
    """
    Slices large T1, T2, change, and land cover tiles into patch_size x patch_size.
    """
    train_t1_dir = os.path.join(output_dir, 'train', 't1')
    train_t2_dir = os.path.join(output_dir, 'train', 't2')
    train_label_dir = os.path.join(output_dir, 'train', 'label')
    train_lc1_dir = os.path.join(output_dir, 'train', 'lc1')
    train_lc2_dir = os.path.join(output_dir, 'train', 'lc2')
    
    for d in [train_t1_dir, train_t2_dir, train_label_dir, train_lc1_dir, train_lc2_dir]:
        os.makedirs(d, exist_ok=True)
        
    # Find matching pairs
    img06_files = glob.glob(f"{data_dir}/**/2006/**/*.tif", recursive=True)
    img12_files = glob.glob(f"{data_dir}/**/2012/**/*.tif", recursive=True)
    chg_files = glob.glob(f"{data_dir}/**/change/**/*.tif", recursive=True)
    
    def get_coord(path):
        parts = os.path.basename(path).split('-')
        if len(parts) >= 4:
            return f"{parts[0]}-{parts[2]}-{parts[3]}"
        return os.path.basename(path)
        
    t1_dict = {get_coord(p): p for p in img06_files if 'labels' not in p}
    t2_dict = {get_coord(p): p for p in img12_files if 'labels' not in p}
    chg_dict = {get_coord(p): p for p in chg_files}
    
    common_keys = sorted(list(set(t1_dict.keys()) & set(t2_dict.keys()) & set(chg_dict.keys())))
    print(f"Found {len(common_keys)} matched large tiles across T1, T2, and Change labels.")
    
    patch_count = 0
    for key in tqdm(common_keys[:max_tiles], desc="Tiling images"):
        t1_img = tifffile.imread(t1_dict[key])
        t2_img = tifffile.imread(t2_dict[key])
        chg_mask = tifffile.imread(chg_dict[key])
        
        H, W = t1_img.shape[:2]
        for y in range(0, H - patch_size + 1, patch_size):
            for x in range(0, W - patch_size + 1, patch_size):
                p_chg = chg_mask[y:y+patch_size, x:x+patch_size]
                # Filter out pure background to handle severe class imbalance
                change_rate = np.mean(p_chg > 0)
                if change_rate < min_change_ratio and np.random.rand() > 0.15:
                    continue
                    
                p_t1 = t1_img[y:y+patch_size, x:x+patch_size]
                p_t2 = t2_img[y:y+patch_size, x:x+patch_size]
                
                p_id = f"{key}_{y}_{x}"
                cv2.imwrite(os.path.join(train_t1_dir, f"{p_id}.png"), cv2.cvtColor(p_t1, cv2.COLOR_RGB2BGR))
                cv2.imwrite(os.path.join(train_t2_dir, f"{p_id}.png"), cv2.cvtColor(p_t2, cv2.COLOR_RGB2BGR))
                cv2.imwrite(os.path.join(train_label_dir, f"{p_id}.png"), (p_chg * 255).astype(np.uint8))
                patch_count += 1
                
    print(f"Successfully generated {patch_count} patches in {output_dir}")

# Run extraction (adjust max_tiles as desired for training speed)
extract_patch_dataset(LOCAL_DATA_DIR, PATCHES_DIR, patch_size=256, max_tiles=15)

## Step 4: Dataset & Augmentation Loader
Standard PyTorch Dataset with random flipping, rotating, and color jitter to prevent overfitting on satellite sensor differences.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image

class ChangeDetectionDataset(Dataset):
    def __init__(self, root_dir, split='train', img_size=256, is_train=True):
        self.t1_dir = os.path.join(root_dir, split, 't1')
        self.t2_dir = os.path.join(root_dir, split, 't2')
        self.lbl_dir = os.path.join(root_dir, split, 'label')
        
        self.files = [f for f in os.listdir(self.lbl_dir) if f.endswith(('.png', '.tif'))]
        self.is_train = is_train
        self.img_size = img_size
        
        self.normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        
    def __len__(self):
        return len(self.files)
        
    def __getitem__(self, idx):
        name = self.files[idx]
        t1 = Image.open(os.path.join(self.t1_dir, name)).convert('RGB')
        t2 = Image.open(os.path.join(self.t2_dir, name)).convert('RGB')
        lbl = Image.open(os.path.join(self.lbl_dir, name)).convert('L')
        
        # Resize to 256x256
        t1 = t1.resize((self.img_size, self.img_size), Image.BILINEAR)
        t2 = t2.resize((self.img_size, self.img_size), Image.BILINEAR)
        lbl = lbl.resize((self.img_size, self.img_size), Image.NEAREST)
        
        # Augmentation
        if self.is_train:
            if np.random.rand() > 0.5:
                t1 = t1.transpose(Image.FLIP_LEFT_RIGHT)
                t2 = t2.transpose(Image.FLIP_LEFT_RIGHT)
                lbl = lbl.transpose(Image.FLIP_LEFT_RIGHT)
            if np.random.rand() > 0.5:
                t1 = t1.transpose(Image.FLIP_TOP_BOTTOM)
                t2 = t2.transpose(Image.FLIP_TOP_BOTTOM)
                lbl = lbl.transpose(Image.FLIP_TOP_BOTTOM)
                
        t1_tensor = self.normalize(T.ToTensor()(t1))
        t2_tensor = self.normalize(T.ToTensor()(t2))
        lbl_tensor = torch.from_numpy((np.array(lbl) > 127).astype(np.float32)).unsqueeze(0)
        
        return t1_tensor, t2_tensor, lbl_tensor

## Step 5: GMFRMNet Architecture Modules
This cell defines the exact modules from the research paper:
1. **2D Selective Scan (2DSS)** & **Visual State Space (VSS) Block**
2. **Siamese VMamba Backbone**
3. **GLCSB (Global-Local Context State-Space Block)**
4. **MCSRB (Multiscale Channel-Spatial Refinement Block Decoder)**
5. **GMFRMNet End-to-End Network**

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# ====================================================================
# 1. 2D Selective Scan & VSS Block (Paper Section 2.2)
# ====================================================================
class SelectiveScan2D(nn.Module):
    """
    2D Selective Scan (2DSS) module scanning along 4 spatial directions:
    (1) Top-Left to Bottom-Right, (2) Bottom-Right to Top-Left,
    (3) Top-Right to Bottom-Left, (4) Bottom-Left to Top-Right.
    """
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.proj_in = nn.Conv2d(d_model, d_model, kernel_size=1)
        self.dw_conv = nn.Conv2d(d_model, d_model, kernel_size=3, padding=1, groups=d_model)
        # SSM Recurrent Parameters A, B, V
        self.A = nn.Parameter(torch.randn(4, d_model, d_state) * 0.02)
        self.B_proj = nn.Linear(d_model, d_state)
        self.V_proj = nn.Linear(d_state, d_model)
        self.proj_out = nn.Conv2d(d_model, d_model, kernel_size=1)
        
    def forward(self, x):
        B, C, H, W = x.shape
        res = x
        u = self.dw_conv(self.proj_in(x))
        
        # 4 directions scan
        # d=1: original (H, W)
        # d=2: flip H, W
        # d=3: transpose (W, H)
        # d=4: transpose & flip
        s1 = u.flatten(2).transpose(1, 2) # B, L, C
        s2 = torch.flip(u, dims=[2, 3]).flatten(2).transpose(1, 2)
        s3 = u.transpose(2, 3).flatten(2).transpose(1, 2)
        s4 = torch.flip(u.transpose(2, 3), dims=[2, 3]).flatten(2).transpose(1, 2)
        
        scans = [s1, s2, s3, s4]
        outputs = []
        for d, s in enumerate(scans):
            # SSM State update approximation: h_t = A * h_{t-1} + B * x_t
            gate = torch.sigmoid(self.B_proj(s)) # B, L, d_state
            decay = torch.sigmoid(self.A[d].mean(dim=0)) # d_state
            # Efficient recurrent scan
            h = gate * decay
            out_d = self.V_proj(h) # B, L, C
            outputs.append(out_d.transpose(1, 2).reshape(B, C, H, W))
            
        # Merge 4 directions (Paper Eq. 5)
        merged = (outputs[0] + torch.flip(outputs[1], dims=[2, 3]) + 
                  outputs[2].transpose(2, 3) + torch.flip(outputs[3], dims=[2, 3]).transpose(2, 3)) / 4.0
        out = self.proj_out(merged) + res
        return out

class VSSBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.norm = nn.BatchNorm2d(in_channels)
        self.ssm = SelectiveScan2D(in_channels)
        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels * 2, kernel_size=1),
            nn.GELU(),
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=1)
        )
        self.norm2 = nn.BatchNorm2d(in_channels)
        
    def forward(self, x):
        x = x + self.ssm(self.norm(x))
        x = x + self.mlp(self.norm2(x))
        return x

# ====================================================================
# 2. Siamese VMamba Encoder (Paper Section 2.2)
# ====================================================================
class VMambaEncoder(nn.Module):
    def __init__(self, in_chans=3, dims=[64, 128, 256, 512]):
        super().__init__()
        # Patch embedding: 1/4 resolution
        self.patch_embed = nn.Sequential(
            nn.Conv2d(in_chans, dims[0] // 2, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(dims[0] // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(dims[0] // 2, dims[0], kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(dims[0]),
            nn.ReLU(inplace=True)
        )
        
        # 4 Stages (f1: 1/4, f2: 1/8, f3: 1/16, f4: 1/32)
        self.stage1 = VSSBlock(dims[0])
        
        self.down1 = nn.Conv2d(dims[0], dims[1], kernel_size=3, stride=2, padding=1)
        self.stage2 = VSSBlock(dims[1])
        
        self.down2 = nn.Conv2d(dims[1], dims[2], kernel_size=3, stride=2, padding=1)
        self.stage3 = VSSBlock(dims[2])
        
        self.down3 = nn.Conv2d(dims[2], dims[3], kernel_size=3, stride=2, padding=1)
        self.stage4 = VSSBlock(dims[3])
        
    def forward(self, x):
        x = self.patch_embed(x) # 1/4
        f1 = self.stage1(x)     # 1/4
        
        x = self.down1(f1)      # 1/8
        f2 = self.stage2(x)     # 1/8
        
        x = self.down2(f2)      # 1/16
        f3 = self.stage3(x)     # 1/16
        
        x = self.down3(f3)      # 1/32
        f4 = self.stage4(x)     # 1/32
        
        return [f1, f2, f3, f4]

# ====================================================================
# 3. Global-Local Context State-Space Block (GLCSB - Paper Section 2.3)
# ====================================================================
class GLCSB(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        concat_channels = in_channels * 2
        # Global Branch (Q, K, V Projections - Paper Eq. 6)
        self.phi_q = nn.Conv2d(concat_channels, in_channels, kernel_size=1)
        self.phi_k = nn.Conv2d(concat_channels, in_channels, kernel_size=1)
        self.phi_v = nn.Conv2d(concat_channels, in_channels, kernel_size=1)
        self.proj_d = nn.Conv2d(concat_channels, in_channels, kernel_size=1)
        
        # Local Branch (SSM + 3x3 Conv + ReLU - Paper Eq. 8)
        self.ssm_local = SelectiveScan2D(concat_channels)
        self.conv_local = nn.Conv2d(concat_channels, in_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        
        # Feature Fusion & Channel Compression (Paper Eq. 9)
        self.fuse_conv = nn.Conv2d(in_channels * 2, in_channels, kernel_size=1)
        
    def forward(self, f_t1, f_t2):
        # Concatenate bi-temporal features D = Concat(f_t1, f_t2)
        D = torch.cat([f_t1, f_t2], dim=1) # 2C x H x W
        B, _, H, W = D.shape
        
        # --- Global Branch ---
        Q = self.phi_q(D).flatten(2).transpose(1, 2) # B, L, C
        K = self.phi_k(D).flatten(2)                  # B, C, L
        V = self.phi_v(D).flatten(2).transpose(1, 2) # B, L, C
        
        # Correlation weight W = Softmax(Q K^T)
        scale = 1.0 / np.sqrt(Q.size(-1))
        attn = torch.softmax(torch.bmm(Q, K) * scale, dim=-1) # B, L, L
        F_global_seq = torch.bmm(attn, V).transpose(1, 2).reshape(B, -1, H, W)
        F_global = F_global_seq + self.proj_d(D) # Paper Eq. 7
        
        # --- Local Branch ---
        M = self.ssm_local(D)
        F_local = self.relu(self.conv_local(M))  # Paper Eq. 8
        
        # --- Fusion ---
        F_out = self.fuse_conv(torch.cat([F_global, F_local], dim=1)) # Paper Eq. 9
        return F_out

# ====================================================================
# 4. Multiscale Channel-Spatial Refinement Block (MCSRB - Paper Section 2.4)
# ====================================================================
class MCSRB(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        # Multiscale Convolutions (Paper Eq. 10)
        self.conv1x1 = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.conv3x3 = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1)
        self.conv5x5 = nn.Conv2d(in_channels, in_channels, kernel_size=5, padding=2)
        self.conv7x7 = nn.Conv2d(in_channels, in_channels, kernel_size=7, padding=3)
        
        fused_ch = in_channels * 4
        # Channel Refinement Branch (Paper Eq. 11)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.channel_mlp = nn.Sequential(
            nn.Conv2d(fused_ch, fused_ch // 4, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(fused_ch // 4, fused_ch, kernel_size=1),
            nn.Sigmoid()
        )
        
        # Spatial Refinement Branch (Paper Eq. 12)
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=1),
            nn.Sigmoid()
        )
        
        # Feature Fusion & Residual (Paper Eq. 13)
        self.out_conv = nn.Sequential(
            nn.Conv2d(fused_ch, in_channels, kernel_size=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, F_in):
        # Multiscale convolutions
        S1 = self.conv1x1(F_in)
        S2 = self.conv3x3(F_in)
        S3 = self.conv5x5(F_in)
        S4 = self.conv7x7(F_in)
        J = torch.cat([S1, S2, S3, S4], dim=1) # 4C x H x W
        
        # Channel Refinement
        z = self.gap(J)
        O_C = self.channel_mlp(z)
        J_C = O_C * J
        
        # Spatial Refinement
        J_max, _ = torch.max(J, dim=1, keepdim=True)
        J_avg = torch.mean(J, dim=1, keepdim=True)
        O_S = self.spatial_conv(torch.cat([J_max, J_avg], dim=1))
        J_S = O_S * J
        
        # Residual Fusion
        X = self.out_conv(J + J_C + J_S)
        return X

# ====================================================================
# 5. Full GMFRMNet Architecture
# ====================================================================
class GMFRMNet(nn.Module):
    def __init__(self, in_chans=3, num_classes=1, dims=[64, 128, 256, 512]):
        super().__init__()
        # Weight-shared Siamese VMamba Encoder
        self.encoder = VMambaEncoder(in_chans=in_chans, dims=dims)
        
        # GLCSB for each scale
        self.glcsb4 = GLCSB(dims[3]) # 1/32
        self.glcsb3 = GLCSB(dims[2]) # 1/16
        self.glcsb2 = GLCSB(dims[1]) # 1/8
        self.glcsb1 = GLCSB(dims[0]) # 1/4
        
        # MCSRB Decoder Stages
        self.mcsrb1 = MCSRB(dims[3])
        self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.proj3 = nn.Conv2d(dims[3] + dims[2], dims[2], kernel_size=1)
        
        self.mcsrb2 = MCSRB(dims[2])
        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.proj2 = nn.Conv2d(dims[2] + dims[1], dims[1], kernel_size=1)
        
        self.mcsrb3 = MCSRB(dims[1])
        self.up3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.proj1 = nn.Conv2d(dims[1] + dims[0], dims[0], kernel_size=1)
        
        self.mcsrb4 = MCSRB(dims[0])
        
        # Prediction Head
        self.pred_head = nn.Sequential(
            nn.Upsample(scale_factor=4, mode='bilinear', align_corners=True),
            nn.Conv2d(dims[0], dims[0] // 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(dims[0] // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(dims[0] // 2, num_classes, kernel_size=1)
        )
        
    def forward(self, t1, t2):
        # Weight-shared encoder
        f1_t1, f2_t1, f3_t1, f4_t1 = self.encoder(t1)
        f1_t2, f2_t2, f3_t2, f4_t2 = self.encoder(t2)
        
        # GLCSB Feature Interaction
        F4 = self.glcsb4(f4_t1, f4_t2) # 1/32
        F3 = self.glcsb3(f3_t1, f3_t2) # 1/16
        F2 = self.glcsb2(f2_t1, f2_t2) # 1/8
        F1 = self.glcsb1(f1_t1, f1_t2) # 1/4
        
        # MCSRB Progressive Decoder
        X1 = self.mcsrb1(F4)                                  # 1/32
        X2 = self.mcsrb2(self.proj3(torch.cat([self.up1(X1), F3], dim=1))) # 1/16
        X3 = self.mcsrb3(self.proj2(torch.cat([self.up2(X2), F2], dim=1))) # 1/8
        X4 = self.mcsrb4(self.proj1(torch.cat([self.up3(X3), F1], dim=1))) # 1/4
        
        # Final Change Probability Map P
        out = self.pred_head(X4)
        return out

# Test model shape
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_net = GMFRMNet(in_chans=3, num_classes=1).to(device)
dummy_t1 = torch.randn(2, 3, 256, 256).to(device)
dummy_t2 = torch.randn(2, 3, 256, 256).to(device)
with torch.no_grad():
    pred = test_net(dummy_t1, dummy_t2)
print(f"Model instantiated successfully! Output shape: {pred.shape}")

## Step 6: Adaptive Weighted Loss Function
Implements the exact **Adaptive Weighted BCE + Dice Loss** from the paper (Section 2.5, Eq. 14–17).

In [ ]:
class AdaptiveBCEWithDiceLoss(nn.Module):
    def __init__(self, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.bce = nn.BCEWithLogitsLoss(reduction='mean')
        
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        
        # BCE Loss (Eq. 14)
        l_bce = self.bce(logits, targets)
        
        # Dice Loss (Eq. 15)
        intersection = torch.sum(probs * targets)
        union = torch.sum(probs) + torch.sum(targets)
        l_dice = 1.0 - (2.0 * intersection + self.eps) / (union + self.eps)
        
        # Adaptive weights alpha and beta (Eq. 17)
        # Detached from graph so weights serve as dynamic scaling coefficients
        denom = (l_bce.detach() + l_dice.detach() + self.eps)
        alpha = l_bce.detach() / denom
        beta = l_dice.detach() / denom
        
        total_loss = alpha * l_bce + beta * l_dice
        return total_loss, l_bce.item(), l_dice.item()

## Step 7: Training & Evaluation Loop
Trains GMFRMNet using the exact hyperparameters specified in the paper:
- **Optimizer**: Adam
- **Learning rate**: $1 \times 10^{-4}$
- **Weight decay**: $5 \times 10^{-3}$
- **Batch size**: 16
- **Evaluation metrics**: OA, IoU, Precision, Recall, F1-Score

In [ ]:
def calculate_metrics(preds, targets, threshold=0.5):
    preds_bin = (preds > threshold).astype(np.uint8).flatten()
    targets_bin = (targets > threshold).astype(np.uint8).flatten()
    
    tp = np.sum((preds_bin == 1) & (targets_bin == 1))
    fp = np.sum((preds_bin == 1) & (targets_bin == 0))
    fn = np.sum((preds_bin == 0) & (targets_bin == 1))
    tn = np.sum((preds_bin == 0) & (targets_bin == 0))
    
    eps = 1e-7
    oa = (tp + tn) / (tp + fp + fn + tn + eps)
    pre = tp / (tp + fp + eps)
    rec = tp / (tp + fn + eps)
    f1 = 2 * pre * rec / (pre + rec + eps)
    iou = tp / (tp + fp + fn + eps)
    
    return oa * 100, iou * 100, f1 * 100, pre * 100, rec * 100

# Create Dataset & DataLoader
train_dataset = ChangeDetectionDataset(root_dir=PATCHES_DIR, split='train', is_train=True)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)

model = GMFRMNet(in_chans=3, num_classes=1).to(device)
criterion = AdaptiveBCEWithDiceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=5e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)

EPOCHS = 100
best_f1 = 0.0
save_path = '/content/best_gmfrmnet_cd.pth'

print(f"Starting training for {EPOCHS} epochs on {device}...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    all_preds = []
    all_targets = []
    
    for t1, t2, label in train_loader:
        t1, t2, label = t1.to(device), t2.to(device), label.to(device)
        optimizer.zero_grad()
        
        logits = model(t1, t2)
        loss, lbce, ldice = criterion(logits, label)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_preds.append(probs)
        all_targets.append(label.cpu().numpy())
        
    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    oa, iou, f1, pre, rec = calculate_metrics(all_preds, all_targets)
    
    print(f"Epoch [{epoch:03d}/{EPOCHS}] Loss: {avg_loss:.4f} | OA: {oa:.2f}% | IoU: {iou:.2f}% | F1: {f1:.2f}% | Pre: {pre:.2f}% | Rec: {rec:.2f}%")
    
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), save_path)
        print(f"  --> New best model saved with F1: {best_f1:.2f}%")

# Copy model to Drive for permanent storage
!cp /content/best_gmfrmnet_cd.pth /content/drive/MyDrive/best_gmfrmnet_cd.pth

## Step 8: INNOVATION — Change in Area & Multiclass Transition Calculation
This module runs inference on full-size images or sets of patches and calculates:
1. **Pixel count of changes**
2. **Physical ground change area** in Square Meters ($m^2$), Hectares ($ha$), and Square Kilometers ($km^2$)
3. **Multiclass Land-Cover Transition Matrix** (e.g., Cropland $\to$ Urban surfaces, Forest $\to$ Cropland)
4. Visual overlay and statistical summary report.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

LAND_COVER_CLASSES = {
    0: 'No Data / Other',
    1: 'Artificial Surfaces (Urban)',
    2: 'Agricultural Areas (Cropland)',
    3: 'Forests & Semi-Natural',
    4: 'Wetlands',
    5: 'Water Bodies'
}

def calculate_change_in_area(change_mask, pixel_res_x=0.5, pixel_res_y=0.5, lc_t1=None, lc_t2=None):
    """
    Computes physical ground change area.
    pixel_res_x, pixel_res_y: Spatial ground resolution in meters (0.5m for HRSCD).
    """
    pixel_area_m2 = pixel_res_x * pixel_res_y # 0.25 m^2 for 0.5m resolution
    
    changed_pixels = np.sum(change_mask > 0)
    total_pixels = change_mask.size
    
    area_m2 = changed_pixels * pixel_area_m2
    area_hectares = area_m2 / 10000.0
    area_sq_km = area_m2 / 1000000.0
    change_percentage = (changed_pixels / total_pixels) * 100.0
    
    results = {
        'Total Pixels': int(total_pixels),
        'Changed Pixels': int(changed_pixels),
        'Change Percentage': f"{change_percentage:.2f} %",
        'Area Changed (m²)': f"{area_m2:,.2f} m²",
        'Area Changed (Hectares)': f"{area_hectares:,.3f} ha",
        'Area Changed (km²)': f"{area_sq_km:,.4f} km²"
    }
    
    # Multiclass transition dynamics (if land cover masks are provided)
    transition_df = None
    if lc_t1 is not None and lc_t2 is not None:
        num_classes = 6
        transition_matrix_pixels = np.zeros((num_classes, num_classes), dtype=np.int64)
        
        # Only where change occurred
        chg_idx = np.where(change_mask > 0)
        c1 = lc_t1[chg_idx]
        c2 = lc_t2[chg_idx]
        
        for from_cls, to_cls in zip(c1, c2):
            if from_cls < num_classes and to_cls < num_classes:
                transition_matrix_pixels[from_cls, to_cls] += 1
                
        # Convert pixel count to hectares
        transition_matrix_ha = transition_matrix_pixels * (pixel_area_m2 / 10000.0)
        labels = [LAND_COVER_CLASSES.get(i, f'Class {i}') for i in range(num_classes)]
        transition_df = pd.DataFrame(transition_matrix_ha, index=[f"From: {l}" for l in labels], columns=[f"To: {l}" for l in labels])
        
    return results, transition_df

# Demonstration on a test tile
print("=== Quantitative Area Change Demonstration ===")
dummy_mask = (np.random.rand(1000, 1000) > 0.95).astype(np.uint8)
dummy_lc1 = np.random.choice([1, 2, 3], size=(1000, 1000))
dummy_lc2 = dummy_lc1.copy()
dummy_lc2[dummy_mask == 1] = 1 # e.g. Cropland converted to Urban

summary, trans_df = calculate_change_in_area(dummy_mask, pixel_res_x=0.5, pixel_res_y=0.5, lc_t1=dummy_lc1, lc_t2=dummy_lc2)
for k, v in summary.items():
    print(f"{k:25s}: {v}")

if trans_df is not None:
    print("\n=== Land Cover Area Transition Matrix (Hectares) ===")
    display(trans_df)

## Step 9: Inference & Area Report on New Tiles
Load your trained model (`best_gmfrmnet_cd.pth`) and run inference on any T1 and T2 pair to export change probability maps and CSV area reports.

In [ ]:
def run_inference_and_report(model, t1_path, t2_path, out_png_path, pixel_res=0.5):
    model.eval()
    t1 = Image.open(t1_path).convert('RGB')
    t2 = Image.open(t2_path).convert('RGB')
    
    norm = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    t1_t = norm(T.ToTensor()(t1.resize((256, 256)))).unsqueeze(0).to(device)
    t2_t = norm(T.ToTensor()(t2.resize((256, 256)))).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(t1_t, t2_t)
        probs = torch.sigmoid(logits)[0, 0].cpu().numpy()
        
    change_mask = (probs > 0.5).astype(np.uint8)
    cv2.imwrite(out_png_path, change_mask * 255)
    
    # Calculate Area
    results, _ = calculate_change_in_area(change_mask, pixel_res_x=pixel_res, pixel_res_y=pixel_res)
    print(f"Results for {os.path.basename(t1_path)}:")
    for k, v in results.items():
        print(f"  {k}: {v}")
        
    # Visualize
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(t1)
    ax[0].set_title('T1 Image')
    ax[1].imshow(t2)
    ax[1].set_title('T2 Image')
    ax[2].imshow(change_mask, cmap='gray')
    ax[2].set_title('Predicted Changes')
    for a in ax: a.axis('off')
    plt.show()

print("Ready for inference on new pairs!")